In [1]:
import logging
from kiteconnect import KiteConnect
import datetime
logging.basicConfig(level=logging.INFO)
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import pyotp
import requests
import dotenv
from os import environ as env
dotenv.load_dotenv(dotenv.find_dotenv())

True

In [10]:
INDEX_CSV = "/Users/sarthak/Downloads/MW-NIFTY-BANK-08-Dec-2025.csv"

In [2]:
UserId: str = env["USER_ID"]
Password: str = env["PASSWORD"]
ApiKey: str = env["API_KEY"]
ApiSecret: str = env["API_SECRET"]
TOTP_SECRET: str = env["TOTP_SECRET"]

totp = pyotp.TOTP(TOTP_SECRET)
kite = KiteConnect(api_key=ApiKey)

In [3]:
def get_request_token(user_id: str, password: str, kite: KiteConnect, totp: pyotp.TOTP) -> str:
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

    with requests.Session() as session:
        # 1) post credentials
        login_payload = {"user_id": user_id, "password": password, "type": "user_id"}
        login_resp = session.post("https://kite.zerodha.com/api/login", data=login_payload, headers=headers).json()

        # 2) post TOTP
        totp_payload = {
            "user_id": user_id,
            "request_id": login_resp["data"]["request_id"],
            "twofa_type": "totp",
            "skip_session": True,
            "twofa_value": f"{totp.now()}",
        }
        session.post("https://kite.zerodha.com/api/twofa", data=totp_payload, headers=headers)

        # 3) complete connect/login flow and extract request_token
        connect_resp = session.get(kite.login_url(), allow_redirects=False)
        finish_resp = session.get(connect_resp.headers["location"], allow_redirects=False)

        return parse_qs(urlparse(finish_resp.headers["location"]).query)["request_token"][0]

In [4]:
# may need to re autheticate if expired
REQUEST_TOKEN = get_request_token(UserId, Password, kite, totp)

In [6]:
data = kite.generate_session(REQUEST_TOKEN, api_secret=ApiSecret)
kite.set_access_token(data["access_token"])

In [16]:
index_df = pd.read_csv(INDEX_CSV)
index_df.columns = [i.strip() for i in index_df.columns]
index_df.head()

,SYMBOL,OPEN,HIGH,LOW,PREV. CLOSE,LTP,INDICATIVE CLOSE,CHNG,%CHNG,VOLUME \n(shares),VALUE \n (₹ Crores),52W H,52W L,30 D %CHNG
0,NIFTY BANK,"59,133.20","59,806.60","59,106.55","59,288.70","59,777.20",-,488.50,0.82,"11,94,81,620","5,949.73","60,114.30","47,702.90",3.37
1,SBIN,948.85,973.30,946.70,948.10,971.70,-,23.60,2.49,"1,73,47,847","1,668.12",999.00,680.00,1.45
2,PNB,119.69,121.90,119.15,119.53,121.62,-,2.09,1.75,"2,26,67,150",274.25,127.80,85.46,-1.25
3,BANKBARODA,287.10,294.00,286.55,288.20,292.00,-,3.80,1.32,"73,76,690",214.42,303.95,190.70,1.56
4,AUBANK,948.90,965.00,946.10,948.75,960.90,-,12.15,1.28,"16,40,399",157.25,966.90,478.35,9.13


In [20]:
ins_map = {i["tradingsymbol"]: i["instrument_token"] for i in kite.instruments(exchange="NSE")}

In [48]:
def get_all_time_data(ins_token, interval, to_date):
    interval_limit = {
        'minute' : 60,
        '3minute' : 100,
        '5minute' : 100,
        '10minute' : 100,
        '15minute' : 200,
        '30minute' : 200,
        '60minute' : 400,
        'day' : 2000
    }

    to_date = datetime.datetime(to_date.year, to_date.month, to_date.day)
    from_date = to_date - datetime.timedelta(days=interval_limit[interval])

    output = []

    try:
        data = kite.historical_data(ins_token, from_date=from_date, to_date=to_date, interval=interval)
        while len(data) != 0:
            output.extend(data)
            to_date = from_date
            from_date = to_date - datetime.timedelta(days=interval_limit[interval])
            data = kite.historical_data(ins_token, from_date=from_date, to_date=to_date, interval=interval)
    except Exception as e:
        raise e

    return output

In [59]:
for _, ins in index_df.iterrows():
    symb = ins.SYMBOL
    ins_token = ins_map[symb]

    data = get_all_time_data(ins_token, "minute", datetime.datetime.today())
    
    df = pd.DataFrame.from_records(data)
    df.to_csv(f"{symb}_ohlc.csv")


In [17]:
?kite.historical_data

Signature:
kite.historical_data(
    instrument_token,
    from_date,
    to_date,
    interval,
    continuous=False,
    oi=False,
)
Docstring:
Retrieve historical data (candles) for an instrument.

Although the actual response JSON from the API does not have field
names such has 'open', 'high' etc., this function call structures
the data into an array of objects with field names. For example:

- `instrument_token` is the instrument identifier (retrieved from the instruments()) call.
- `from_date` is the From date (datetime object or string in format of yyyy-mm-dd HH:MM:SS.
- `to_date` is the To date (datetime object or string in format of yyyy-mm-dd HH:MM:SS).
- `interval` is the candle interval (minute, day, 5 minute etc.).
- `continuous` is a boolean flag to get continuous data for futures and options instruments.
- `oi` is a boolean flag to get open interest.
File:      ~/src/algo-trade/venv/lib/python3.14/site-packages/kiteconnect/connect.py
Type:      method

In [33]:
out = kite.historical_data(194325505, from_date=datetime.datetime(2025, 12, 1), to_date=datetime.datetime(2025, 12, 5), interval="minute")

In [50]:
import csv

In [45]:
datetime.datetime.today().day

8

In [53]:
write = csv.writer(open('tmp', 'w'))

In [56]:
df = pd.DataFrame.from_records(out[:5])

In [57]:
?df.to_csv

Signature:
df.to_csv(
    path_or_buf: 'FilePath | WriteBuffer[bytes] | WriteBuffer[str] | None' = None,
    *,
    sep: 'str' = ',',
    na_rep: 'str' = '',
    float_format: 'str | Callable | None' = None,
    columns: 'Sequence[Hashable] | None' = None,
    header: 'bool_t | list[str]' = True,
    index: 'bool_t' = True,
    index_label: 'IndexLabel | None' = None,
    mode: 'str' = 'w',
    encoding: 'str | None' = None,
    compression: 'CompressionOptions' = 'infer',
    quoting: 'int | None' = None,
    quotechar: 'str' = '"',
    lineterminator: 'str | None' = None,
    chunksize: 'int | None' = None,
    date_format: 'str | None' = None,
    doublequote: 'bool_t' = True,
    escapechar: 'str | None' = None,
    decimal: 'str' = '.',
    errors: 'OpenFileErrors' = 'strict',
    storage_options: 'StorageOptions | None' = None,
) -> 'str | None'
Docstring:
Write object to a comma-separated values (csv) file.

Parameters
----------
path_or_buf : str, path object, file-like object,